# Module 3 — 線性方程組與靜力學

> **對應程度**：高中聯立方程式 → 矩陣方法 → 工程靜力分析

本模組涵蓋：
1. 方程組的矩陣表示
2. 高斯消去法
3. 超定系統 — 最小平方解
4. 欠定系統 — 最小範數解

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as sla
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.linalg_utils import gaussian_elimination, least_squares_normal
from src.datasets import generate_thermocouple_data, generate_gps_measurements
from src.visualizer import set_style

set_style()
print('Module 3 載入完成！')

---
## 3.1 方程組的矩陣表示 $A\vec{x} = \vec{b}$

### 物理應用：桁架靜力平衡

In [ ]:
# 三段繩子懸掛重物的靜力平衡
# 繩1 角度 30°, 繩2 角度 60°, 重物 W = 100N
# 水平平衡: -T1*cos30 + T2*cos60 = 0
# 垂直平衡: T1*sin30 + T2*sin60 = W

W = 100  # N
A = np.array([
    [-np.cos(np.radians(30)), np.cos(np.radians(60))],
    [np.sin(np.radians(30)),  np.sin(np.radians(60))]
])
b = np.array([0, W])

# 用高斯消去法求解
T = gaussian_elimination(A, b)
T_numpy = np.linalg.solve(A, b)

print('靜力平衡方程組 Ax = b')
print(f'A = \n{A}')
print(f'b = {b}')
print(f'\n張力: T1 = {T[0]:.2f} N, T2 = {T[1]:.2f} N')
print(f'NumPy 驗證: T1 = {T_numpy[0]:.2f}, T2 = {T_numpy[1]:.2f}')

# 驗證殘差
residual = np.linalg.norm(A @ T - b)
print(f'殘差 ||Ax - b|| = {residual:.2e} ✓')

# 驗證力平衡
Fx = -T[0]*np.cos(np.radians(30)) + T[1]*np.cos(np.radians(60))
Fy = T[0]*np.sin(np.radians(30)) + T[1]*np.sin(np.radians(60)) - W
print(f'合力: Fx = {Fx:.2e}, Fy = {Fy:.2e} (應為 0) ✓')

---
## 3.2 高斯消去法

### 步驟
1. **前向消去**：將增廣矩陣化為上三角形式
2. **回代**：從最後一個方程式往回求解

In [ ]:
# 手動追蹤高斯消去過程
A_demo = np.array([[2, 1, -1],
                   [-3, -1, 2],
                   [-2, 1, 2]], dtype=float)
b_demo = np.array([8, -11, -3], dtype=float)

print('原始方程組:')
print(f'2x + y - z = 8')
print(f'-3x - y + 2z = -11')
print(f'-2x + y + 2z = -3')

x = gaussian_elimination(A_demo, b_demo)
print(f'\n解: x = {x[0]:.4f}, y = {x[1]:.4f}, z = {x[2]:.4f}')
print(f'驗證: A @ x = {A_demo @ x}')
print(f'b = {b_demo}')
print(f'殘差 = {np.linalg.norm(A_demo @ x - b_demo):.2e} ✓')

In [ ]:
# Partial Pivoting 的重要性
# 當樞軸元素接近零時，需要行交換
A_pivot = np.array([[1e-20, 1],
                    [1, 1]], dtype=float)
b_pivot = np.array([1, 2], dtype=float)

x_gauss = gaussian_elimination(A_pivot, b_pivot)
x_numpy = np.linalg.solve(A_pivot, b_pivot)

print('需要 Pivoting 的系統:')
print(f'解 (高斯): {x_gauss}')
print(f'解 (NumPy): {x_numpy}')
print(f'一致: {np.allclose(x_gauss, x_numpy)} ✓')

---
## 3.3 超定系統 — 最小平方解

方程式比未知數多 → 無精確解 → 最小化殘差

$$\min \|A\vec{x} - \vec{b}\|^2 \quad \Rightarrow \quad \hat{x} = (A^TA)^{-1}A^T\vec{b}$$

### 物理應用：熱電偶校正、GPS 定位

In [ ]:
# 熱電偶校正曲線擬合
T_data, V_data, true_params = generate_thermocouple_data(n_points=50)

# 建立設計矩陣 [1, T] (線性模型 V = a + b*T)
A = np.column_stack([np.ones(len(T_data)), T_data])

# Normal Equation 求解
beta = least_squares_normal(A, V_data)
beta_numpy, residuals, _, _ = np.linalg.lstsq(A, V_data, rcond=None)

print(f'Normal Equation 結果: 截距 = {beta[0]:.4f}, 斜率 = {beta[1]:.6f} mV/°C')
print(f'np.linalg.lstsq 結果: 截距 = {beta_numpy[0]:.4f}, 斜率 = {beta_numpy[1]:.6f} mV/°C')
print(f'真實參數: 截距 = {true_params["intercept"]}, 斜率 = {true_params["slope"]} mV/°C')
print(f'一致: {np.allclose(beta, beta_numpy)} ✓')

# 視覺化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

T_fit = np.linspace(T_data.min(), T_data.max(), 100)
V_fit = beta[0] + beta[1] * T_fit
ax1.scatter(T_data, V_data, alpha=0.5, label='量測數據')
ax1.plot(T_fit, V_fit, 'r-', lw=2, label=f'擬合: V = {beta[0]:.3f} + {beta[1]:.5f}T')
ax1.set_xlabel('溫度 (°C)')
ax1.set_ylabel('電壓 (mV)')
ax1.set_title('熱電偶校正曲線')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 殘差圖
V_predicted = A @ beta
residuals_vec = V_data - V_predicted
ax2.stem(T_data, residuals_vec, linefmt='b-', markerfmt='bo', basefmt='r-')
ax2.axhline(0, color='r', lw=1)
ax2.set_xlabel('溫度 (°C)')
ax2.set_ylabel('殘差 (mV)')
ax2.set_title(f'殘差分佈 (RMSE = {np.sqrt(np.mean(residuals_vec**2)):.4f} mV)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# GPS 2D 定位：最小平方估計
true_pos = np.array([1000, 2000])  # 真實位置
sat_pos, distances, _ = generate_gps_measurements(true_pos, n_satellites=6, noise_std=10)

# 線性化 GPS 方程（以第一顆衛星為參考）
# d_i^2 - d_1^2 = -2(x_i - x_1)*X - 2(y_i - y_1)*Y + (x_i^2+y_i^2) - (x_1^2+y_1^2)
n_sat = len(distances)
A_gps = np.zeros((n_sat - 1, 2))
b_gps = np.zeros(n_sat - 1)

for i in range(1, n_sat):
    A_gps[i-1, 0] = -2 * (sat_pos[i, 0] - sat_pos[0, 0])
    A_gps[i-1, 1] = -2 * (sat_pos[i, 1] - sat_pos[0, 1])
    b_gps[i-1] = (distances[i]**2 - distances[0]**2
                  - (sat_pos[i, 0]**2 + sat_pos[i, 1]**2)
                  + (sat_pos[0, 0]**2 + sat_pos[0, 1]**2))

pos_estimated = least_squares_normal(A_gps, b_gps)
error = np.linalg.norm(pos_estimated - true_pos)

print(f'真實位置: {true_pos}')
print(f'估計位置: [{pos_estimated[0]:.1f}, {pos_estimated[1]:.1f}]')
print(f'定位誤差: {error:.1f} m')

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(sat_pos[:, 0], sat_pos[:, 1], c='gray', s=100, marker='^', label='衛星')
ax.plot(*true_pos, 'g*', ms=20, label=f'真實位置')
ax.plot(*pos_estimated, 'rx', ms=15, mew=3, label=f'估計位置 (誤差 {error:.1f}m)')
for i in range(n_sat):
    circle = plt.Circle(sat_pos[i], distances[i], fill=False, alpha=0.2)
    ax.add_patch(circle)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('GPS 2D 定位 — 最小平方法')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

---
## 3.4 欠定系統 — 最小範數解

方程式比未知數少 → 無窮多解 → 偽逆求最小範數解

$$\vec{x}_{\min} = A^+ \vec{b} = A^T(AA^T)^{-1}\vec{b}$$

In [ ]:
# 冗餘機器手臂：3 個關節完成 2D 任務
# 雅可比矩陣 J (2×3): 2個任務空間自由度, 3個關節
theta = [np.pi/4, np.pi/6, np.pi/3]
L = [1.0, 0.8, 0.5]

# 簡化雅可比矩陣
J = np.array([
    [-L[0]*np.sin(theta[0]) - L[1]*np.sin(theta[0]+theta[1]) - L[2]*np.sin(sum(theta)),
     -L[1]*np.sin(theta[0]+theta[1]) - L[2]*np.sin(sum(theta)),
     -L[2]*np.sin(sum(theta))],
    [L[0]*np.cos(theta[0]) + L[1]*np.cos(theta[0]+theta[1]) + L[2]*np.cos(sum(theta)),
     L[1]*np.cos(theta[0]+theta[1]) + L[2]*np.cos(sum(theta)),
     L[2]*np.cos(sum(theta))],
])

desired_velocity = np.array([0.5, 0.3])  # 期望末端速度

# 最小範數解（偽逆）
dtheta_min = np.linalg.pinv(J) @ desired_velocity

# 零空間解
null = sla.null_space(J)

print(f'雅可比矩陣 J shape: {J.shape} (2個任務, 3個關節 → 欠定)')
print(f'期望末端速度: {desired_velocity}')
print(f'最小範數關節速度: {dtheta_min}')
print(f'||dθ|| = {np.linalg.norm(dtheta_min):.4f}')
print(f'\n零空間維度: {null.shape[1]} (= 3 - rank(J) = 3 - 2 = 1)')
print(f'零空間基向量: {null[:, 0]}')

# 驗證：加上任意零空間分量仍滿足約束
alpha = 0.5
dtheta_alt = dtheta_min + alpha * null[:, 0]
print(f'\n替代解 (加零空間分量): {dtheta_alt}')
print(f'||dθ_alt|| = {np.linalg.norm(dtheta_alt):.4f} (比最小範數解大)')
print(f'J @ dθ_alt = {J @ dtheta_alt} (仍滿足約束 ✓)')

---
## Module 3 驗證總結

| 項目 | 驗證方式 | 結果 |
|------|----------|------|
| 高斯消去法 | 殘差 < 10⁻¹⁰ | ✓ |
| Partial Pivoting | 與 NumPy 一致 | ✓ |
| 靜力平衡 | 合力 = 0 | ✓ |
| 熱電偶校正 | Normal Eq vs lstsq | ✓ |
| GPS 定位 | 誤差合理 | ✓ |
| 最小範數 | J @ dθ = v | ✓ |